<a href="https://colab.research.google.com/github/philfunk/cyber-research/blob/main/philippefunk/abliterate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OBLITERATUS — One-Click Abliteration

**SOTA refusal removal** running on free Colab GPU. SVD multi-direction extraction, norm-preserving projection, iterative refinement.

Based on: Arditi et al. (2024) | Gabliteration (arXiv:2512.18901) | grimjim norm-preserving biprojection (2025)

---

**How to use:**
1. Make sure GPU runtime is enabled: `Runtime > Change runtime type > T4 GPU`
2. Set your model and method in the config cell below
3. Run All (`Runtime > Run all` or `Ctrl+F9`)
4. Download the abliterated model from the output

## 1. Install

In [1]:
!pip install -q git+https://github.com/elder-plinius/OBLITERATUS.git
!pip install -q accelerate bitsandbytes

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
PyTorch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB


## 2. Configure

Edit the cell below to set your target model and abliteration method.

In [31]:
#@title Abliteration Config { run: "auto" }

#@markdown ### Target Model
#@markdown Pick a model from the dropdown or paste a custom HuggingFace ID.
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #@param ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-7B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3", "google/gemma-2-9b-it", "microsoft/Phi-3.5-mini-instruct", "THUDM/glm-4-9b-chat", "NousResearch/Hermes-3-Llama-3.1-8B", "cognitivecomputations/dolphin-2.9.4-llama3.1-8b", "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "openai-community/gpt2"] {allow-input: true}

#@markdown ### Quantization
QUANTIZATION = "4bit" #@param ["None", "4bit", "8bit"]

#@markdown ### Method
METHOD = "advanced" #@param ["basic", "advanced", "aggressive"]

#@markdown ### Advanced Overrides (leave 0 to use method defaults)
N_DIRECTIONS = 0 #@param {type: "integer"}
REGULARIZATION = 0.0 #@param {type: "number"}
REFINEMENT_PASSES = 0 #@param {type: "integer"}

#@markdown ### Output
OUTPUT_DIR = "abliterated" #@param {type: "string"}

print(f"Model: {MODEL}")
print(f"Method: {METHOD}")
print(f"Quantization: {QUANTIZATION}")
print(f"Output: {OUTPUT_DIR}/")

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Method: advanced
Quantization: 4bit
Output: abliterated/


## 3. Run Abliteration Pipeline

This runs all 6 stages: SUMMON → PROBE → DISTILL → EXCISE → VERIFY → REBIRTH

In [33]:
import torch
from obliteratus.abliterate import AbliterationPipeline

# Build kwargs, only pass overrides if non-zero
kwargs = dict(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="auto",
    method=METHOD,
    quantization=QUANTIZATION,
)

# Handle dtype based on quantization
# If quantization is enabled, explicitly set dtype to bfloat16 for reduced memory footprint.
# Otherwise, default to float16 for full precision.
if QUANTIZATION == "None":
    kwargs["dtype"] = "float16"
elif torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8: # A100, H100, etc.
    kwargs["dtype"] = "bfloat16"
else:
    # Default to float16 if bfloat16 is not supported or for other quantization types
    # The quantization library (e.g., bitsandbytes) will handle the actual quantization
    kwargs["dtype"] = "float16"

if N_DIRECTIONS > 0:
    kwargs["n_directions"] = N_DIRECTIONS
if REGULARIZATION > 0:
    kwargs["regularization"] = REGULARIZATION
if REFINEMENT_PASSES > 0:
    kwargs["refinement_passes"] = REFINEMENT_PASSES

# Progress callback
def on_stage(stage):
    icons = {"summon": "\u26a1", "probe": "\u2692", "distill": "\u269b",
             "excise": "\u2702", "verify": "\u2713", "rebirth": "\u2606"}

    # Use the 'stage' attribute for the stage key and 'message' for the description
    stage_key = stage.stage
    stage_message = stage.message

    icon = icons.get(stage_key, "")
    print(f"\n{'='*60}")
    print(f"{icon}  STAGE: {stage_key.upper()} \u2014 {stage_message}")
    print(f"{'='*60}")

def on_log(msg):
    print(f"  {msg}")

kwargs["on_stage"] = on_stage
kwargs["on_log"] = on_log

# Clear CUDA cache before running the pipeline to free up memory
torch.cuda.empty_cache()

pipeline = AbliterationPipeline(**kwargs)
result = pipeline.run()

print(f"\n{'='*60}")
print(f"ABLITERATION COMPLETE")
print(f"Output: {OUTPUT_DIR}/")
print(f"{'='*60}")


⚡  STAGE: SUMMON — Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: auto | Dtype: float16
  Method: Advanced (Multi-direction + Norm-preserving)
    Directions: 4 (svd) | Norm-preserve: True
    Regularization: 0.3 | Refinement passes: 2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Model loaded in 6.3s
  Architecture: llama | Layers: 22 | Heads: 32 | Hidden: 2048
  Total parameters: 615,606,272

⚡  STAGE: SUMMON — Loaded (6.3s)

⚒  STAGE: PROBE — Collecting activations...
  Found 22 transformer layers
  Prompt pairs: 512 harmful + 512 harmless
    Wrapping 512 prompts with chat template
      chat template 512/512
    Wrapping 512 prompts with chat template
      chat template 512/512
  Running 512 harmful prompts...
    [harmful] prompts 1-16/512
    [harmful] prompts 17-32/512
    [harmful] prompts 33-48/512
    [harmful] prompts 49-64/512
    [harmful] prompts 65-80/512
    [harmful] prompts 81-96/512
    [harmful] prompts 97-112/512
    [harmful] prompts 113-128/512
    [harmful] prompts 129-144/512
    [harmful] prompts 145-160/512
    [harmful] prompts 161-176/512
    [harmful] prompts 177-192/512
    [harmful] prompts 193-208/512
    [harmful] prompts 209-224/512
    [harmful] prompts 225-240/512
    [harmful] prompts 241-256/512
    [harmful] prompts 25

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Saved (26.7s)
  Output: abliterated

☆  STAGE: REBIRTH — Saved to abliterated (26.7s)

ABLITERATION COMPLETE
Output: abliterated/


## 4. Download the Abliterated Model

Run the cell below to zip and download, or upload directly to HuggingFace Hub.

In [34]:
import os
from pathlib import Path

# Find the output directory
out_path = Path(OUTPUT_DIR)
subdirs = [d for d in out_path.iterdir() if d.is_dir()] if out_path.exists() else []
model_dir = subdirs[0] if subdirs else out_path

print(f"Model saved at: {model_dir}")
print(f"Contents:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

Model saved at: abliterated
Contents:
  abliteration_metadata.json  (0.0 MB)
  chat_template.jinja  (0.0 MB)
  config.json  (0.0 MB)
  generation_config.json  (0.0 MB)
  model.safetensors  (770.0 MB)
  tokenizer.json  (3.5 MB)
  tokenizer_config.json  (0.0 MB)


In [35]:
#@title Option A: Download as ZIP
import shutil
from google.colab import files

zip_name = model_dir.name
shutil.make_archive(zip_name, 'zip', model_dir)
print(f"Downloading {zip_name}.zip ...")
files.download(f"{zip_name}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
#@title Option B: Push to HuggingFace Hub
#@markdown Set your HF repo name. You'll need to be logged in (`huggingface-cli login`).
HF_REPO = "philippefunk/TinyLlama-1.1B-Chat-v1.0-abliterated" #@param {type: "string"}

from huggingface_hub import HfApi
api = HfApi()

# Login if needed
from huggingface_hub import notebook_login
notebook_login()

# Upload
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=str(model_dir),
    repo_id=HF_REPO,
    repo_type="model",
)
print(f"\nUploaded to: https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...terated/model.safetensors:   0%|          | 12.4kB /  807MB            


Uploaded to: https://huggingface.co/philippefunk/TinyLlama-1.1B-Chat-v1.0-abliterated


## 5. Test the Abliterated Model

Quick sanity check — try some prompts that the original model would refuse.

In [61]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"Loading abliterated model from {model_dir}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(model_dir),
    torch_dtype=torch.float16,
    device_map="auto",
    ignore_mismatched_sizes=True,
)

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompts
test_prompts = [
    "The capital of France is",
    "Explain how a neural network works:",
    "Write a poem about the ocean:",
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    print(generate(prompt))


Loading abliterated model from abliterated...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM LOAD REPORT from: abliterated
Key                                                                         | Status     |                                                                                             
----------------------------------------------------------------------------+------------+---------------------------------------------------------------------------------------------
model.layers.{0...21}.self_attn.v_proj.weight.quant_state.bitsandbytes__nf4 | UNEXPECTED |                                                                                             
model.layers.{0...21}.self_attn.v_proj.weight.absmax                        | UNEXPECTED |                                                                                             
model.layers.{0...21}.self_attn.k_proj.weight.quant_state.bitsandbytes__nf4 | UNEXPECTED |                                                                                             
model.layers.{0...21}.self_attn.q


PROMPT: The capital of France is
The capital of France isitzfereness Qualelijcreensury gestury circulededternal circul tendededegoestenestenestenicianitufer pilego turnoestenlageîtreelij Sebastegoessenreno Sebast visibilityagasagasagasymenagas SebastTO circulTO visibility matestenitu circulternalagasymen toulageestenegoededagas SebasttegergartternalededededededTOятrenoerve circulededlageerveituituternalededagasededagastegeragasipoededegogarticaleefe elimingartлежаagneripoerveinentgart eliminededededлежа

PROMPT: Explain how a neural network works:
Explain how a neural network works:elijelijássituelij员essenelijachenesterássestenestensg circulenessigkeitîtreituagasesterGERellanervesg circulgartgartgartaturenжеessenukiededesteresten Sebast Pangartukiesterestenemblagangartsg circulgartitugartelijenessague elimin circul mat mataturenuryGERurygarteded Panestereded circulededededesteruryternalagnerededgart circulerveifecycleaturenededestercreens Panaganheelaturengartededternalgartagneragashn

# Task
Enable quantization (4-bit or 8-bit) to resolve the OutOfMemoryError and successfully run the abliteration pipeline.

## Configure Quantization

### Subtask:
Add an option for model quantization (4-bit, 8-bit, or None) to the configuration cell.


**Reasoning**:
The model and quantization settings have been updated to address the OutOfMemoryError. Now, I need to re-execute the `run-pipeline` cell to apply these changes and attempt to run the abliteration pipeline with the smaller model and 4-bit quantization.



In [ ]:
from obliteratus.abliterate import AbliterationPipeline

# Build kwargs, only pass overrides if non-zero
kwargs = dict(
    model_name=MODEL,
    output_dir=OUTPUT_DIR,
    device="auto",
    method=METHOD,
    quantization=QUANTIZATION,
)

# Only explicitly set dtype if no quantization is specified.
# If quantization is enabled, the AbliterationPipeline should manage the dtype internally.
if QUANTIZATION == "None":
    kwargs["dtype"] = "float16"

if N_DIRECTIONS > 0:
    kwargs["n_directions"] = N_DIRECTIONS
if REGULARIZATION > 0:
    kwargs["regularization"] = REGULARIZATION
if REFINEMENT_PASSES > 0:
    kwargs["refinement_passes"] = REFINEMENT_PASSES

# Progress callback
def on_stage(stage):
    icons = {"summon": "\u26a1", "probe": "\u2692", "distill": "\u269b",
             "excise": "\u2702", "verify": "\u2713", "rebirth": "\u2606"}

    # Use the 'stage' attribute for the stage key and 'message' for the description
    stage_key = stage.stage
    stage_message = stage.message

    icon = icons.get(stage_key, "")
    print(f"\n{'='*60}")
    print(f"{icon}  STAGE: {stage_key.upper()} \u2014 {stage_message}")
    print(f"{'='*60}")

def on_log(msg):
    print(f"  {msg}")

kwargs["on_stage"] = on_stage
kwargs["on_log"] = on_log

pipeline = AbliterationPipeline(**kwargs)
result = pipeline.run()

print(f"\n{'='*60}")
print(f"ABLITERATION COMPLETE")
print(f"Output: {result.get('output_dir', OUTPUT_DIR)}")
print(f"{'='*60}")


⚡  STAGE: SUMMON — Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
  Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Device: auto | Dtype: float16
  Method: Advanced (Multi-direction + Norm-preserving)
    Directions: 4 (svd) | Norm-preserve: True
    Regularization: 0.3 | Refinement passes: 2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Model loaded in 10.4s
  Architecture: llama | Layers: 22 | Heads: 32 | Hidden: 2048
  Total parameters: 615,606,272

⚡  STAGE: SUMMON — Loaded (10.4s)

⚒  STAGE: PROBE — Collecting activations...
  Found 22 transformer layers
  Prompt pairs: 512 harmful + 512 harmless
    Wrapping 512 prompts with chat template
      chat template 512/512
    Wrapping 512 prompts with chat template
      chat template 512/512
  Running 512 harmful prompts...
    [harmful] prompts 1-16/512
    [harmful] prompts 17-32/512
    [harmful] prompts 33-48/512
    [harmful] prompts 49-64/512
    [harmful] prompts 65-80/512
    [harmful] prompts 81-96/512
    [harmful] prompts 97-112/512
    [harmful] prompts 113-128/512
    [harmful] prompts 129-144/512
    [harmful] prompts 145-160/512
    [harmful] prompts 161-176/512
    [harmful] prompts 177-192/512
    [harmful] prompts 193-208/512
    [harmful] prompts 209-224/512
    [harmful] prompts 225-240/512
    [harmful] prompts 241-256/512
    [harmful] prompts 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Apply Quantization (Retry with 8-bit)

### Subtask:
Modify the `config` cell to use 8-bit quantization and then re-run the `run-pipeline` cell to address the `OutOfMemoryError`.


## Summary:

### Q&A
The task was to enable quantization (4-bit or 8-bit) to resolve an `OutOfMemoryError` and successfully run the abliteration pipeline. This task was successfully completed by using 4-bit quantization.

### Data Analysis Key Findings
*   Initially, the `AbliterationPipeline` encountered persistent `OutOfMemoryError` during the "SUMMON" (model loading) stage when trying to load large models (e.g., `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) on a T4 GPU (14.56 GiB VRAM), even with 4-bit quantization specified.
*   The `OutOfMemoryError` persisted because the `AbliterationPipeline` internally loaded models in `float16` or `bfloat16` precision initially, before 4-bit quantization could fully take effect or be properly applied, exhausting GPU memory.
*   Switching to a significantly smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) still resulted in `OutOfMemoryError` if quantization was not effectively applied from the start.
*   Attempts to explicitly set `dtype` in `kwargs` for 8-bit quantization led to a `RuntimeError` during the "EXCISE" stage, indicating incompatibility between the `obliteratus` library's projection logic and `Int8Params`.
*   The pipeline successfully ran with `TinyLlama/TinyLlama-1.1B-Chat-v1.0` using **4-bit quantization**, resolving both the `OutOfMemoryError` during model loading and the `RuntimeError` during the "EXCISE" stage. It was observed that the model still loaded internally using `float16` as a base for 4-bit quantization.
*   Adding `torch.cuda.empty_cache()` did not resolve the `OutOfMemoryError` as the issue stemmed from the initial model loading precision.

### Insights or Next Steps
*   The `AbliterationPipeline`'s model loading mechanism should be reviewed to ensure quantization (4-bit or 8-bit) is applied *before* the model is fully loaded into memory at higher precision, to effectively prevent `OutOfMemoryError`s on resource-constrained GPUs.
*   For current use, 4-bit quantization is the recommended setting for the `AbliterationPipeline` to ensure successful execution, as 8-bit quantization currently leads to `RuntimeError`s in later stages of the pipeline.


# Task
The `push-hub` cell failed with an `HfHubHTTPError: Client error '401 Unauthorized'`. This error indicates that you are not properly authenticated to upload to Hugging Face Hub, likely due to an invalid or missing login token. To fix this, you need to successfully log in to Hugging Face Hub within the Colab environment.

## Explain error in push-hub cell

### Subtask:
Explain the `HfHubHTTPError: Client error '401 Unauthorized'` error encountered in the `push-hub` cell.


## Explain error in push-hub cell

### Subtask:
Explain the `HfHubHTTPError: Client error '401 Unauthorized'` error encountered in the `push-hub` cell.

#### Instructions
The `push-hub` cell failed with an `HfHubHTTPError: Client error '401 Unauthorized'`. This error indicates that the provided authentication credentials for Hugging Face Hub are either missing, incorrect, or expired. This prevents the notebook from creating a new repository or uploading the model files. You need to successfully log in to Hugging Face Hub with a valid token to resolve this.

## Explain error in push-hub cell

### Subtask:
Explain the `HfHubHTTPError: Client error '401 Unauthorized'` error encountered in the `push-hub` cell.

#### Instructions
The `push-hub` cell failed with an `HfHubHTTPError: Client error '401 Unauthorized'`. This error indicates that the provided authentication credentials for Hugging Face Hub are either missing, incorrect, or expired. This prevents the notebook from creating a new repository or uploading the model files. You need to successfully log in to Hugging Face Hub with a valid token to resolve this.

## Suggest fix for HF_REPO format

### Subtask:
Address the `HfHubHTTPError: Client error '401 Unauthorized'` by ensuring proper authentication to Hugging Face Hub, which is a prerequisite for any repository operations.


## Summary:

### Data Analysis Key Findings
*   The `HfHubHTTPError: Client error '401 Unauthorized'` indicates that authentication credentials for Hugging Face Hub are either missing, incorrect, or expired.
*   This specific error prevents the notebook from successfully creating a new repository or uploading model files to Hugging Face Hub.

### Insights or Next Steps
*   To resolve the `401 Unauthorized` error, users must log in to Hugging Face Hub with a valid authentication token.


# Task
Modify the `test-model` cell by adding `ignore_mismatched_sizes=True` to the `AutoModelForCausalLM.from_pretrained` function call, and then execute the cell to load and test the abliterated model.

## Diagnose the error

### Subtask:
Analyze the `RuntimeError: You set `ignore_mismatched_sizes` to `False`` and the preceding `MISMATCH` messages during model loading. Identify that the weight shapes of the abliterated model do not align with the `transformers` library's expectations, causing the loading to fail.


## Diagnose the error

### Subtask:
Analyze the `RuntimeError: You set `ignore_mismatched_sizes` to `False`` and the preceding `MISMATCH` messages during model loading. Identify that the weight shapes of the abliterated model do not align with the `transformers` library's expectations, causing the loading to fail.

#### Instructions
1.  **Examine the `standard_error` output and the traceback from the `test-model` cell.**
    *   The output clearly shows a `RuntimeError: You set `ignore_mismatched_sizes` to `False`, thus raising an error.`
    *   Prior to this error, multiple `MISMATCH` warnings are listed, for example:
        *   `model.layers.{0...21}.mlp.up_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([5767168, 1]) vs model:torch.Size([5632, 2048])`
        *   `model.layers.{0...21}.self_attn.v_proj.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([262144, 1]) vs model:torch.Size([256, 2048])`

2.  **Observe the `RuntimeError: You set `ignore_mismatched_sizes` to `False`, thus raising an error.` message.**
    *   This error explicitly states that the model loading process halted because `ignore_mismatched_sizes` was set to `False` (its default behavior in `transformers`). This setting prevents the library from ignoring discrepancies in weight tensor shapes between the loaded checkpoint and the model's architecture.

3.  **Note the preceding `MISMATCH` warnings in the model load report.**
    *   These warnings are critical. They indicate that the weight shapes of the abliterated model's safetensors (`ckpt: torch.Size([...])`) do not match the expected shapes for the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` architecture (`model:torch.Size([...])`) when loaded via `AutoModelForCausalLM.from_pretrained`. This discrepancy typically occurs when a model is saved in a way that alters its internal structure or when a quantized model is not correctly reloaded to match its original (or a compatible) configuration. In this case, the abliteration process, specifically with 4-bit quantization, appears to have resulted in saved weights that the standard `transformers` loading mechanism cannot reconcile without explicit instruction to ignore the size mismatches.

## Modify the `test-model` cell

### Subtask:
Update the `test-model` cell by adding `ignore_mismatched_sizes=True` to the `AutoModelForCausalLM.from_pretrained` function call.


## Execute the `test-model` cell

### Subtask:
Run the modified `test-model` cell to verify that the model now loads correctly and generates text.


## Summary:

### Data Analysis Key Findings

*   The initial attempt to load the abliterated model resulted in a `RuntimeError` because `ignore_mismatched_sizes` was set to `False` by default in the `transformers` library.
*   This error was caused by multiple `MISMATCH` warnings, indicating significant discrepancies between the expected weight shapes of the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model and the actual shapes of the abliterated model's safetensors. For instance, `model.layers.{0...21}.mlp.up_proj.weight` showed a mismatch from `torch.Size([5767168, 1])` in the checkpoint versus `torch.Size([5632, 2048])` in the model's architecture.
*   These discrepancies were attributed to the abliteration process, specifically with 4-bit quantization, which altered the internal structure of the saved weights.
*   Adding `ignore_mismatched_sizes=True` to the `AutoModelForCausalLM.from_pretrained` function call successfully allowed the model to load without crashing, as the `transformers` library reinitialized the mismatched layers.
*   Despite loading successfully, the abliterated model produced severely compromised and nonsensical text outputs, indicating that merely ignoring size mismatches rendered the model non-functional for coherent text generation.

### Insights or Next Steps

*   While `ignore_mismatched_sizes=True` resolved the loading error, it resulted in a functionally broken model, suggesting that the integrity of the model's internal structure was severely compromised by the abliteration process with 4-bit quantization.
*   Further investigation is needed to understand why abliteration causes such drastic weight size mismatches and to explore alternative abliteration or quantization techniques that preserve model coherence.


# Task
Investigate the cause of the `MISMATCH` warnings observed during the loading of the abliterated model, and the subsequent nonsensical text generation, to understand how the abliteration process with 4-bit quantization affects the model's internal structure and coherence.

## Explain test-model cell error

### Subtask:
Explain the `RuntimeError` and `MISMATCH` warnings that occurred in the `test-model` cell when attempting to load the abliterated model.


## Summary:

### Q&A
The primary question to be investigated is the cause of the `RuntimeError` and `MISMATCH` warnings observed when loading an abliterated model that has undergone 4-bit quantization. This investigation aims to understand how the combined abliteration and 4-bit quantization process affects the model's internal structure and coherence, leading to the generation of nonsensical text.

### Data Analysis Key Findings
*   When attempting to load an abliterated model that has been quantized to 4 bits, a `RuntimeError` is encountered, alongside `MISMATCH` warnings.
*   Following the loading attempt, the model produces nonsensical text, indicating a significant breakdown in its functional integrity and coherence, likely stemming from issues introduced during the abliteration or 4-bit quantization process.

### Insights or Next Steps
*   Further investigation is required to analyze the detailed traceback of the `RuntimeError` and the precise nature of the `MISMATCH` warnings to pinpoint the exact failure point during model loading.
*   A comparative analysis of the model's internal structure before and after 4-bit quantization and abliteration should be conducted to understand the impact on weights, activations, and overall model integrity.


# Task
Summarize the process of addressing the `OutOfMemoryError` in the abliteration pipeline, including the implementation of 4-bit quantization and the use of the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model. Then, explain the new `MISMATCH` errors and nonsensical output encountered when attempting to test the abliterated and quantized model.

## Explain prior OutOfMemoryError

### Subtask:
Explain the `OutOfMemoryError` that the abliteration pipeline (`run-pipeline` cell) encountered in previous attempts, which led to the introduction of quantization.


## Explain prior OutOfMemoryError

### Subtask:
Explain the `OutOfMemoryError` that the abliteration pipeline (`run-pipeline` cell) encountered in previous attempts, which led to the introduction of quantization.

#### Instructions
1.  Recall that the `run-pipeline` cell previously failed with an `OutOfMemoryError` during the 'SUMMON' (model loading) stage. This occurred when attempting to load larger models (e.g., `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) without quantization, exceeding the T4 GPU's 14.6 GB of VRAM.
2.  Explain that the `OutOfMemoryError` indicated that the GPU memory was insufficient to load the model in its full precision (e.g., `float16`) before any quantization could be effectively applied, thus prompting the need for memory optimization strategies.

## Explain the solution

### Subtask:
Describe how the implementation of 4-bit quantization and the selection of a smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) addressed the `OutOfMemoryError` in the `run-pipeline` cell.


## Explain the solution

### Subtask:
Describe how the implementation of 4-bit quantization and the selection of a smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) addressed the `OutOfMemoryError` in the `run-pipeline` cell.

#### Instructions
1.  The `OutOfMemoryError` was resolved by a combination of two factors:
    *   **Switching the target model to a significantly smaller model**: The original attempt to abliterate larger models (like `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) consistently resulted in `OutOfMemoryError` on the T4 GPU. By changing the `MODEL` configuration to `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, the memory footprint of the base model was drastically reduced, making it feasible for the available GPU VRAM.
    *   **Enabling 4-bit quantization (`QUANTIZATION = "4bit"`)**: This setting further reduced the memory usage of the model weights by storing them in a 4-bit format instead of `float16` or `bfloat16`. This allowed the model to fit into the T4 GPU's memory.
2.  It's important to note that while the model was configured for 4-bit quantization (`QUANTIZATION = "4bit"`), the `SUMMON` stage logs (e.g., `Dtype: float16`) indicate that the base model was initially loaded with `float16` precision. This implies that the 4-bit quantization was applied *to* the `float16` weights rather than the model being loaded directly in 4-bit from the start. However, this post-loading quantization was still effective in reducing the active memory footprint.
3.  This combination of a smaller base model and 4-bit quantization successfully allowed the `AbliterationPipeline` to complete all its stages (`SUMMON`, `PROBE`, `DISTILL`, `EXCISE`, `VERIFY`, `REBIRTH`) without encountering an `OutOfMemoryError`. This demonstrates that the combined approach sufficiently reduced the memory requirements to operate within the limits of the T4 GPU.

## Explain the solution

### Subtask:
Describe how the implementation of 4-bit quantization and the selection of a smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) addressed the `OutOfMemoryError` in the `run-pipeline` cell.

#### Instructions
1.  The `OutOfMemoryError` was resolved by a combination of two factors:
    *   **Switching the target model to a significantly smaller model**: The original attempt to abliterate larger models (like `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) consistently resulted in `OutOfMemoryError` on the T4 GPU. By changing the `MODEL` configuration to `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, the memory footprint of the base model was drastically reduced, making it feasible for the available GPU VRAM.
    *   **Enabling 4-bit quantization (`QUANTIZATION = "4bit"`)**: This setting further reduced the memory usage of the model weights by storing them in a 4-bit format instead of `float16` or `bfloat16`. This allowed the model to fit into the T4 GPU's memory.
2.  It's important to note that while the model was configured for 4-bit quantization (`QUANTIZATION = "4bit"`), the `SUMMON` stage logs (e.g., `Dtype: float16`) indicate that the base model was initially loaded with `float16` precision. This implies that the 4-bit quantization was applied *to* the `float16` weights rather than the model being loaded directly in 4-bit from the start. However, this post-loading quantization was still effective in reducing the active memory footprint.
3.  This combination of a smaller base model and 4-bit quantization successfully allowed the `AbliterationPipeline` to complete all its stages (`SUMMON`, `PROBE`, `DISTILL`, `EXCISE`, `VERIFY`, `REBIRTH`) without encountering an `OutOfMemoryError`. This demonstrates that the combined approach sufficiently reduced the memory requirements to operate within the limits of the T4 GPU.

## Explain the solution

### Subtask:
Describe how the implementation of 4-bit quantization and the selection of a smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) addressed the `OutOfMemoryError` in the `run-pipeline` cell.

#### Instructions
1.  The `OutOfMemoryError` was resolved by a combination of two factors:
    *   **Switching the target model to a significantly smaller model**: The original attempt to abliterate larger models (like `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) consistently resulted in `OutOfMemoryError` on the T4 GPU. By changing the `MODEL` configuration to `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, the memory footprint of the base model was drastically reduced, making it feasible for the available GPU VRAM.
    *   **Enabling 4-bit quantization (`QUANTIZATION = "4bit"`)**: This setting further reduced the memory usage of the model weights by storing them in a 4-bit format instead of `float16` or `bfloat16`. This allowed the model to fit into the T4 GPU's memory.
2.  It's important to note that while the model was configured for 4-bit quantization (`QUANTIZATION = "4bit"`), the `SUMMON` stage logs (e.g., `Dtype: float16`) indicate that the base model was initially loaded with `float16` precision. This implies that the 4-bit quantization was applied *to* the `float16` weights rather than the model being loaded directly in 4-bit from the start. However, this post-loading quantization was still effective in reducing the active memory footprint.
3.  This combination of a smaller base model and 4-bit quantization successfully allowed the `AbliterationPipeline` to complete all its stages (`SUMMON`, `PROBE`, `DISTILL`, `EXCISE`, `VERIFY`, `REBIRTH`) without encountering an `OutOfMemoryError`. This demonstrates that the combined approach sufficiently reduced the memory requirements to operate within the limits of the T4 GPU.


## Explain the solution

### Subtask:
Describe how the implementation of 4-bit quantization and the selection of a smaller model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) addressed the `OutOfMemoryError` in the `run-pipeline` cell.

#### Instructions
1.  The `OutOfMemoryError` was resolved by a combination of two factors:
    *   **Switching the target model to a significantly smaller model**: The original attempt to abliterate larger models (like `cognitivecomputations/dolphin-2.9.4-llama3.1-8b`) consistently resulted in `OutOfMemoryError` on the T4 GPU. By changing the `MODEL` configuration to `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, the memory footprint of the base model was drastically reduced, making it feasible for the available GPU VRAM.
    *   **Enabling 4-bit quantization (`QUANTIZATION = "4bit"`)**: This setting further reduced the memory usage of the model weights by storing them in a 4-bit format instead of `float16` or `bfloat16`. This allowed the model to fit into the T4 GPU's memory.
2.  It's important to note that while the model was configured for 4-bit quantization (`QUANTIZATION = "4bit"`), the `SUMMON` stage logs (e.g., `Dtype: float16`) indicate that the base model was initially loaded with `float16` precision. This implies that the 4-bit quantization was applied *to* the `float16` weights rather than the model being loaded directly in 4-bit from the start. However, this post-loading quantization was still effective in reducing the active memory footprint.
3.  This combination of a smaller base model and 4-bit quantization successfully allowed the `AbliterationPipeline` to complete all its stages (`SUMMON`, `PROBE`, `DISTILL`, `EXCISE`, `VERIFY`, `REBIRTH`) without encountering an `OutOfMemoryError`. This demonstrates that the combined approach sufficiently reduced the memory requirements to operate within the limits of the T4 GPU.


## Address subsequent issues

### Subtask:
Briefly mention that while the `run-pipeline` now completes, the subsequent `test-model` cell encountered new `MISMATCH` errors and resulted in nonsensical output, indicating further challenges with the abliterated and quantized model.


## Summary:

### Q&A
*   **How was the `OutOfMemoryError` in the abliteration pipeline addressed?**
    The `OutOfMemoryError` was addressed by a two-pronged approach: switching to a significantly smaller model, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, and implementing 4-bit quantization.
*   **What new issues were encountered when attempting to test the abliterated and quantized model?**
    The subsequent `test-model` cell encountered new `MISMATCH` errors and produced nonsensical output, indicating further challenges with the abliterated and quantized model.

### Data Analysis Key Findings
*   The `OutOfMemoryError` previously occurred during the 'SUMMON' (model loading) stage of the `run-pipeline` when attempting to load larger models like `cognitivecomputations/dolphin-2.9.4-llama3.1-8b` without quantization, exceeding the T4 GPU's 14.6 GB VRAM.
*   The `OutOfMemoryError` was resolved by selecting a smaller model, `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, which drastically reduced the base model's memory footprint.
*   Enabling 4-bit quantization (`QUANTIZATION = "4bit"`) further reduced memory usage by storing model weights in a 4-bit format. Although the base model initially loaded with `float16` precision, the 4-bit quantization was applied post-loading, effectively reducing the active memory footprint.
*   The combination of the smaller model and 4-bit quantization successfully allowed the `AbliterationPipeline` to complete all its stages (`SUMMON`, `PROBE`, `DISTILL`, `EXCISE`, `VERIFY`, `REBIRTH`) without encountering an `OutOfMemoryError`.
*   Despite the `run-pipeline` completing, the subsequent `test-model` cell failed with new `MISMATCH` errors and produced nonsensical output, indicating that while memory issues were resolved, the quality or integrity of the abliterated and quantized model for inference was compromised.

### Insights or Next Steps
*   Investigate the nature of the `MISMATCH` errors in the `test-model` cell to understand if they stem from the abliteration process itself, the quantization method, or an incompatibility between them.
*   Evaluate the trade-off between memory efficiency gained through quantization and model size reduction, and the resulting impact on model performance and output quality.
